# Recomendador basado en contenido — Estrategia 1: TF-IDF

**Introducción al Procesamiento del Lenguaje Natural — TFI 1C 2026**

Esta es la **primera de las dos estrategias de representación**. Implementa un sistema de recomendación *basado en contenido* con **TF-IDF**, una representación léxica, dispersa e interpretable. La segunda estrategia (LLM local) está en `recomendador_llm.ipynb`; el informe compara ambas.

**Decisiones que vienen del EDA:**

- Representamos cada película con **`description` + `keywords` + `genre`** (no solo la sinopsis).
- Modelamos al usuario combinando **historial + query**, que aportan información distinta.
- Toleramos **español + inglés** en el preprocesamiento.
- Usamos el `data/mapa_titulos.csv` generado en el EDA para cruzar el historial con el corpus.


## Anclaje con la materia

Cada celda lleva una nota **📚 Anclaje** con la unidad de origen. Resumen:

- **Preprocesamiento** (regex, minúsculas, stopwords, **stemming `SnowballStemmer`**) → Unidades 1–2 (en particular U2 c3).
- **TF-IDF con `TfidfVectorizer`** → visto en U4 c2 y U5 c4/c5; el concepto (TF, IDF, Zipf) en U2 c2.
- **Vector de documento = promedio de vectores** → U4 c2 ("word embeddings como features": una reseña se representa promediando vectores). Acá promediamos vectores TF-IDF.
- **Similitud coseno (`cosine_similarity`)** → U4 c2 (práctica).
- **n-gramas / bigramas** → U1 (concepto) y U4 (aplicado).

Lo que es **aporte propio** (no hay unidad de *recommender systems*): la **arquitectura de recomendación** (rankear ítems por similitud a un vector de usuario y excluir lo visto), la **combinación ponderada historial+query** con `α`, y la **proxy de evaluación por solapamiento de género**.


## 1. Carga de datos y mapa de títulos

> 📚 **Anclaje — carga (U1) + herramientas de la materia.** `pandas` para el corpus (U1); `TfidfVectorizer` (U4 c2, U5 c4/c5), `cosine_similarity` (U4 c2-práctica) y `SnowballStemmer` (U2 c3) son todas vistas en clase.

In [1]:
import os, re
import numpy as np
import pandas as pd
from unidecode import unidecode
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 80)
DATA = "data"

df = pd.read_csv(os.path.join(DATA, "plots.csv")).reset_index(drop=True)
users = pd.read_csv(os.path.join(DATA, "perfiles_usuarios.csv"))
mapa = pd.read_csv(os.path.join(DATA, "mapa_titulos.csv"), index_col=0)["titulo_corpus"].to_dict()

print(f"Corpus: {len(df)} películas | Usuarios: {len(users)} | Mapa de títulos: {len(mapa)} entradas")

Corpus: 4967 películas | Usuarios: 14 | Mapa de títulos: 57 entradas


## 2. Preprocesamiento

Pipeline léxico clásico (Unidades 1–2): minúsculas, sin acentos, sin puntuación ni dígitos, fuera *stopwords* y **stemming**. Dos decisiones propias de este corpus:

- **Stopwords en español (de la materia) + inglés**, porque las keywords mezclan idiomas.
- **Stemming en lugar de lematización**: el `SnowballStemmer` (visto en U2 c3) no requiere descargar modelos pesados y es suficiente para TF-IDF. *(La materia también lematiza en U4 c1; si querés mayor precisión morfológica podés cambiarlo por lematización en tu máquina.)*

> 📚 **Anclaje — limpieza + stopwords + stemming (Unidades 1, 2 y 2 c3).** Regex de limpieza (U1–U2), lista de stopwords en español **provista por la materia** (U2) y `SnowballStemmer` (U2 c3). La mezcla con stopwords en inglés es por el idioma mixto detectado en el EDA.

In [2]:
# Stopwords en español provistas por la materia (Unidad 2) + inglés (keywords bilingües)
sw_es = set(pd.read_csv(os.path.join(DATA, "stop_words_spanish.csv"))["word"].astype(str))
STOPWORDS = {unidecode(w).lower() for w in sw_es} | set(ENGLISH_STOP_WORDS)
stemmer = SnowballStemmer("spanish")

def preprocesar(texto):
    t = unidecode(str(texto)).lower()
    t = re.sub(r"[^a-z0-9\s]", " ", t)      # solo letras/dígitos/espacios
    t = re.sub(r"\w*\d\w*", " ", t)         # fuera tokens con dígitos
    tokens = [w for w in t.split() if w not in STOPWORDS and len(w) > 2]
    tokens = [stemmer.stem(w) for w in tokens]
    return " ".join(tokens)

# Prueba rápida
print(preprocesar("Una mujer finge su muerte para escapar de su marido controlador."))

muj fing muert escap mar control


### Construcción del documento por película

Combinamos los tres campos. Para que `keywords` y `genre` —más limpios y discriminativos que la sinopsis— pesen más, los **repetimos** antes de vectorizar. Es una forma simple y transparente de subir su influencia sin salir de TF-IDF.

> 📚 **Anclaje — construcción de features de texto (U2 / U4 c2) + decisión propia.** Armar un solo documento por ítem a partir de varios campos es preparación de texto estándar (U2, U4 c2). La **repetición de keywords/genre para ponderar** es una decisión de diseño del grupo.

In [3]:
# Documento por película: sinopsis (x1) + keywords (x2) + genre (x2)
def construir_documento(row):
    desc  = preprocesar(row["description"])
    kw    = preprocesar(row["keywords"])
    gen   = preprocesar(row["genre"])
    return " ".join([desc, kw, kw, gen, gen])

df["doc"] = df.apply(construir_documento, axis=1)
print("Ejemplo de documento procesado:\n")
print(df.loc[2, "name"], "→\n", df.loc[2, "doc"][:300], "...")

Ejemplo de documento procesado:

Durmiendo con su enemigo →
 muj fing muert escap horribl matrimoni descubr impos evit control mar violenci domest muert fing borderlin personality disord relacion madr hij psic thrill violenci domest muert fing borderlin personality disord relacion madr hij psic thrill dram suspens dram suspens ...


## 3. Representación TF-IDF del corpus

Vectorizamos los ~5.000 documentos. `min_df=2` descarta términos que aparecen en una sola película (ruido) y `max_df=0.6` descarta los demasiado frecuentes (poco discriminativos: la lógica de la Ley de Zipf vista en clase). Incluimos bigramas para capturar expresiones como `guerra mundial`.

> 📚 **Anclaje — TF-IDF con `TfidfVectorizer` (U4 c2, U5 c4/c5).** El concepto de TF-IDF y la Ley de Zipf vienen de U2 c2; el `max_df` aplica esa lógica (descartar términos demasiado frecuentes). Los **bigramas** (`ngram_range`) están como concepto en U1 y aplicados en U4.

In [4]:
vectorizer = TfidfVectorizer(min_df=2, max_df=0.6, ngram_range=(1, 2), sublinear_tf=True)
M = vectorizer.fit_transform(df["doc"])     # matriz película × término (dispersa)
print(f"Matriz TF-IDF: {M.shape[0]} películas × {M.shape[1]} términos")

Matriz TF-IDF: 4967 películas × 15520 términos


## 4. Modelado del usuario

El usuario tiene dos fuentes de información de naturaleza distinta:

- **Historial** (qué le gustó): centroide de los vectores TF-IDF de sus películas vistas.
- **Query** (qué quiere ahora): el texto de la query proyectado al **mismo** espacio TF-IDF.

Las combinamos como suma ponderada de vectores **L2-normalizados** (para que ninguna fuente domine por tener más texto):

> `vector_usuario = α · historial_norm + (1 − α) · query_norm`

`α` regula cuánto pesa "lo que vio" frente a "lo que pide". Si el historial quedó vacío (todas las películas ausentes), usamos solo la query, y viceversa. Esto responde directamente a la pregunta de la consigna sobre cómo tratar perfiles con historial degradado.

> 📚 **Anclaje — promedio de vectores (U4 c2) + diseño propio.** Representar al usuario como **centroide** de los vectores de su historial es exactamente la técnica de U4 c2 (representar un texto promediando vectores). **Combinar dos fuentes (historial + query) con un peso `α`** es decisión de diseño de recomendación, sin equivalente directo en clase.

In [5]:
from sklearn.preprocessing import normalize

name_to_idx = {n: i for i, n in enumerate(df["name"])}   # índice interno (evita duplicados de nombre)
hist_cols = [c for c in users.columns if c.startswith("pelicula_")]

def indices_historial(row):
    idxs = []
    for c in hist_cols:
        titulo = row[c]
        if pd.isna(titulo):
            continue
        corpus_title = mapa.get(titulo)          # resuelto en el EDA
        if corpus_title in name_to_idx:
            idxs.append(name_to_idx[corpus_title])
    return idxs

def vector_usuario(row, alpha=0.5):
    # Historial: centroide de las películas vistas presentes en el corpus
    idxs = indices_historial(row)
    if idxs:
        centroide = np.asarray(M[idxs].mean(axis=0)).reshape(1, -1)
        v_hist = normalize(centroide)
    else:
        v_hist = None
    # Query: proyectada al mismo espacio TF-IDF
    q = preprocesar(row["query"])
    v_query_raw = vectorizer.transform([q])
    v_query = normalize(v_query_raw).toarray() if v_query_raw.nnz > 0 else None
    # Combinación
    if v_hist is not None and v_query is not None:
        v = alpha * v_hist + (1 - alpha) * v_query
    elif v_hist is not None:
        v = v_hist
    elif v_query is not None:
        v = v_query
    else:
        v = np.zeros((1, M.shape[1]))
    return np.asarray(v).reshape(1, -1), idxs

# Verificación: cuántas películas de historial recupera cada usuario
for _, r in users.iterrows():
    print(f"{r['nombre']:10} ({r['tipo_perfil']:8}) → {len(indices_historial(r))}/5 películas de historial en el corpus")

Valentina  (definido) → 5/5 películas de historial en el corpus
Rodrigo    (definido) → 5/5 películas de historial en el corpus
Camila     (definido) → 5/5 películas de historial en el corpus
Tomás      (definido) → 5/5 películas de historial en el corpus
Lucía      (definido) → 5/5 películas de historial en el corpus
Martín     (definido) → 5/5 películas de historial en el corpus
Sofía      (definido) → 5/5 películas de historial en el corpus
Diego      (definido) → 5/5 películas de historial en el corpus
Elena      (definido) → 5/5 películas de historial en el corpus
Facundo    (definido) → 5/5 películas de historial en el corpus
Julián     (ambiguo ) → 3/5 películas de historial en el corpus
Mariana    (ambiguo ) → 2/5 películas de historial en el corpus
Nicolás    (ambiguo ) → 5/5 películas de historial en el corpus
Paula      (ambiguo ) → 3/5 películas de historial en el corpus


## 5. Recomendador

Para cada usuario: similitud coseno entre su vector y todas las películas, excluimos las que ya vio (por índice interno) y devolvemos el top-5. Deduplicamos por nombre para no repetir secuelas idénticas.

> 📚 **Anclaje — similitud coseno (U4 c2-práctica) + arquitectura propia.** Rankear por **coseno** entre el vector del usuario y los del corpus usa la métrica vista en clase. La **lógica de recomendación** (excluir lo visto, deduplicar, top-N) es aporte propio: la materia no tiene unidad de sistemas de recomendación.

In [6]:
def recomendar(row, top_n=5, alpha=0.5):
    v, idxs_hist = vector_usuario(row, alpha=alpha)
    sims = cosine_similarity(v, M).flatten()
    excluir = set(idxs_hist)
    orden = np.argsort(-sims)
    recs, vistos = [], set()
    for i in orden:
        if i in excluir:
            continue
        nombre = df.iloc[i]["name"]
        if nombre in vistos:
            continue
        vistos.add(nombre)
        recs.append((nombre, df.iloc[i]["genre"], float(sims[i])))
        if len(recs) == top_n:
            break
    return recs

## 6. Resultados para los 14 usuarios

> 📚 **Anclaje — presentación de resultados.** Recorremos los 14 perfiles y mostramos el top-5; es la corrida del sistema, sin técnica nueva.

In [7]:
for _, r in users.iterrows():
    recs = recomendar(r)
    print(f"\n{'='*78}")
    print(f"{r['nombre']} ({r['tipo_perfil']}) — \"{r['query']}\"")
    print(f"  Historial: {', '.join(str(r[c]) for c in hist_cols)}")
    print(f"  {'-'*74}")
    for nombre, genero, score in recs:
        print(f"   • {nombre[:45]:45} [{genero[:25]:25}] {score:.3f}")


Valentina (definido) — "Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano"
  Historial: Durmiendo con su enemigo, Más allá de la muerte, Desaparecida, ¡Olvídate de mí!, The Crazies
  --------------------------------------------------------------------------
   • Tránsito                                      [drama, misterio, suspense] 0.200
   • Sin City: Una dama por la que matar           [acción, crimen, suspense ] 0.177
   • Lunas de hiel                                 [drama, romance, suspense ] 0.170
   • Un San Valentín de muerte                     [terror, misterio, suspens] 0.159
   • Mujer blanca soltera busca...                 [drama, suspense          ] 0.157

Rodrigo (definido) — "Busco algo basado en hechos reales sobre corrupción o poder político"
  Historial: Adiós Bafana, Mi pie izquierdo, L.A. Confidential, Érase una vez en América, El juego del halcón
  -----------------------------------------------------------------

## 7. Lectura de resultados (dos casos)

Miramos en detalle un caso donde el sistema acierta y uno donde falla, como pide la consigna.

> 📚 **Anclaje — análisis cualitativo de errores (espíritu de U5 c2/c3).** Mirar casos concretos —uno que funciona y uno que no— es el tipo de *análisis de errores* que se hace en las clases de LLMs (U5).

In [8]:
def mostrar(nombre_usuario, alpha=0.5):
    r = users[users["nombre"] == nombre_usuario].iloc[0]
    recs = recomendar(r, alpha=alpha)
    print(f"{r['nombre']} ({r['tipo_perfil']}) — \"{r['query']}\"")
    print(f"Historial: {', '.join(str(r[c]) for c in hist_cols)}\n")
    for nombre, genero, score in recs:
        sin = df[df['name'] == nombre]['description'].iloc[0]
        print(f"• {nombre}  ({score:.3f}) [{genero}]")
        print(f"    {sin[:130]}...")

print(">>> CASO COHERENTE (perfil definido):\n")
mostrar("Rodrigo")

>>> CASO COHERENTE (perfil definido):

Rodrigo (definido) — "Busco algo basado en hechos reales sobre corrupción o poder político"
Historial: Adiós Bafana, Mi pie izquierdo, L.A. Confidential, Érase una vez en América, El juego del halcón

• Gangs of New York  (0.286) [crimen, drama]
    En 1862, Amsterdam Vallon regresa a la zona de Five Points, en Nueva York, en busca de venganza contra Bill el Carnicero, el asesi...
• Buenas noches, y buena suerte.  (0.221) [biografía, drama, historia]
    &quot;Basada en hechos reales, narra la historia del famoso periodista de la CBS Edward R. Murrow (David Strathairn) y su producto...
• 2013: Rescate en L.A.  (0.217) [acción, aventura, ciencia ficción]
    El gobierno de los Estados Unidos recurre de nuevo a Snake Plissken para recuperar un dispositivo que podría acabar con el mundo, ...
• El expreso de Elmira  (0.212) [biografía, drama, deporte]
    ¡Sé protagonista de la inspiradora historia, basada en hechos reales, sobre el primer afroamerica

> 📚 **Anclaje — análisis cualitativo de errores (espíritu de U5 c2/c3).** El contraste con un perfil ambiguo muestra dónde y por qué el sistema falla.

In [9]:
print(">>> CASO DIFÍCIL (perfil ambiguo):\n")
mostrar("Paula")

>>> CASO DIFÍCIL (perfil ambiguo):

Paula (ambiguo) — "No tengo ganas de pensar mucho, pero tampoco quiero algo vacío, algo intermedio"
Historial: Lost in Translation, Transformers, Mamma Mia!, Réquiem por un sueño, Paddington

• Dragon Ball Z  (0.198) [animación, acción, aventura]
    Tras descubrir que es de otro planeta, Goku y sus amigos se ven obligados a defenderlo de un ataque de enemigos extraterrestres....
• El último unicornio  (0.171) [animación, aventura, drama]
    Una valiente unicornio (Mia Farrow) y un mago (Alan Arkin) luchan contra un malvado rey (Sir Christopher Lee) que está obsesionado...
• Savage Grace  (0.158) [drama]
    Sigue el impactante caso de asesinato de Barbara Daly Baekeland, que ocurrió en un lujoso piso de Londres el viernes 17 de noviemb...
• Beautiful Girls  (0.129) [comedia, drama, romance]
    Un pianista en una encrucijada vital regresa a casa, donde sus amigos tienen sus propios problemas....
• Planet 51  (0.127) [animación, aventura, comedia]
 

## 8. Una evaluación cuantitativa simple

Sin *ground truth* no hay métrica de exactitud directa. Pero podemos medir una proxy de **coherencia temática**: ¿cuánto se solapan los géneros de las recomendaciones con los géneros del historial del usuario? Esperamos un solapamiento mayor en los perfiles definidos que en los ambiguos.

> 📚 **Anclaje — evaluar el modelo (U3 / U4 c2) + métrica propia.** La *idea* de evaluar viene de clase (métricas de tópicos en U3, métricas de clasificación en U4 c2). Pero como **no tenemos etiquetas**, no podemos usar accuracy/ROC-AUC de U4 c2; esta **proxy de solapamiento de género** es una heurística propia, con sus límites.

In [10]:
def generos_de(idxs):
    g = set()
    for i in idxs:
        for x in str(df.iloc[i]["genre"]).split(","):
            g.add(x.strip())
    return g

filas = []
for _, r in users.iterrows():
    idxs_hist = indices_historial(r)
    g_hist = generos_de(idxs_hist)
    recs = recomendar(r)
    idxs_rec = [name_to_idx[n] for n, _, _ in recs if n in name_to_idx]
    g_rec = generos_de(idxs_rec)
    solap = len(g_hist & g_rec) / len(g_rec) if g_rec else 0.0
    filas.append((r["nombre"], r["tipo_perfil"], round(solap, 2)))

ev = pd.DataFrame(filas, columns=["nombre", "tipo_perfil", "solapamiento_genero"])
print(ev.to_string(index=False))
print("\nPromedio por tipo de perfil:")
print(ev.groupby("tipo_perfil")["solapamiento_genero"].mean().round(2))

   nombre tipo_perfil  solapamiento_genero
Valentina    definido                 0.71
  Rodrigo    definido                 0.44
   Camila    definido                 1.00
    Tomás    definido                 1.00
    Lucía    definido                 1.00
   Martín    definido                 1.00
    Sofía    definido                 0.71
    Diego    definido                 0.83
    Elena    definido                 1.00
  Facundo    definido                 0.40
   Julián     ambiguo                 0.70
  Mariana     ambiguo                 0.38
  Nicolás     ambiguo                 0.83
    Paula     ambiguo                 0.83

Promedio por tipo de perfil:
tipo_perfil
ambiguo     0.68
definido    0.81
Name: solapamiento_genero, dtype: float64


**Lectura.** El solapamiento de géneros es una proxy imperfecta (un buen "salto" de género puede ser una buena recomendación), pero sirve de diagnóstico: si los perfiles definidos muestran más coherencia que los ambiguos, el sistema se comporta como esperamos. La métrica **no es igualmente válida para todos los perfiles** —en los ambiguos el propio historial es contradictorio— lo que confirma que la evaluación debe leerse por tipo de usuario, no en agregado.

## 9. Límites de esta representación

TF-IDF captura **coincidencia léxica**, no significado: dos películas que cuentan lo mismo con palabras distintas quedan lejos, y la query del usuario ("amenaza invisible", "hacer pensar sobre qué es real") rara vez comparte el vocabulario exacto de las sinopsis. Tampoco entiende negación ni matices ("no muy larga", "distinto a lo de siempre"). Estas limitaciones motivan la segunda estrategia —un LLM local (U5 c3)— que opera sobre el *significado* y no sobre las palabras exactas. Esa comparación es el corazón del informe.

*(Nota de anclaje: una tercera alternativa, también de la materia, sería representar las películas con **Word2Vec** entrenado sobre las sinopsis —U4 c1, la actividad de los Simpsons— y promediar esos embeddings. La descartamos porque con ~5.000 sinopsis cortas los embeddings quedan subentrenados, pero es la opción más "de la casa" para una tercera comparación si hiciera falta.)*
